# Brazilian League Match Predictor — Random Forest

This notebook predicts the outcome (Home Win / Draw / Away Win) of Brazilian Série A championship matches using rolling 5-match form features derived from 9,165 historical matches (2003-2025). The dataset is split temporally: training on seasons through 2022, evaluation on 2023-2025. Random Forest is chosen because it handles non-linear interactions between form features effectively and is robust to overfitting when configured with min_samples_leaf and balanced class weights. Features include rolling goals scored/conceded, win/draw/loss form, season win percentages, and points accumulated over the last five matches per team.

## Setup

In [1]:
import os
os.environ['MPLBACKEND'] = 'agg'
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib-config'

import difflib

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


## Load Data

In [2]:
train = pd.read_parquet('dados/feature_matrix_train.parquet')
test = pd.read_parquet('dados/feature_matrix_test.parquet')

print(f'train shape: {train.shape}')
print(f'test shape:  {test.shape}')


train shape: (8025, 31)
test shape:  (1140, 31)


## Feature Selection

In [3]:
NON_FEATURE_COLS = [
    'id', 'date', 'season', 'round', 'home_team', 'away_team',
    'home_score', 'away_score', 'home_state', 'away_state', 'result',
]
FEATURE_COLS = [c for c in train.columns if c not in NON_FEATURE_COLS]

X_train = train[FEATURE_COLS]
y_train = train['result']
X_test = test[FEATURE_COLS]
y_test = test['result']

print(f'len(FEATURE_COLS) = {len(FEATURE_COLS)}')


len(FEATURE_COLS) = 20


## Model Training

In [4]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    min_samples_leaf=5,
    random_state=42,
)
rf.fit(X_train, y_train)
print('Model trained.')


Model trained.


## Cross-Validation

In [5]:
tss = TimeSeriesSplit(n_splits=5)
cv_acc = cross_val_score(rf, X_train, y_train, cv=tss, scoring='accuracy')
cv_f1 = cross_val_score(rf, X_train, y_train, cv=tss, scoring='f1_macro')

print(f'CV accuracy:  {cv_acc.mean():.3f} +/- {cv_acc.std():.3f}')
print(f'CV macro-F1:  {cv_f1.mean():.3f} +/- {cv_f1.std():.3f}')


CV accuracy:  0.434 +/- 0.016
CV macro-F1:  0.350 +/- 0.012


## Evaluation

In [6]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, labels=['HomeWin', 'Draw', 'AwayWin'], target_names=['HomeWin', 'Draw', 'AwayWin']))


              precision    recall  f1-score   support

     HomeWin       0.52      0.66      0.58       549
        Draw       0.28      0.18      0.22       298
     AwayWin       0.36      0.31      0.33       293

    accuracy                           0.44      1140
   macro avg       0.39      0.38      0.38      1140
weighted avg       0.42      0.44      0.42      1140



In [7]:
labels = ['HomeWin', 'Draw', 'AwayWin']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()


/tmp/ipykernel_75943/3942786054.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Baseline Comparison

In [8]:
naive_acc = (y_test == 'HomeWin').mean()
model_acc = accuracy_score(y_test, y_pred)
model_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Naive (always HomeWin) accuracy:   {naive_acc:.4f}')
print(f'Model test accuracy:               {model_acc:.4f}')
print(f'Model macro-F1:                    {model_f1:.4f}')
print('Note: class_weight=balanced trades raw accuracy for Draw/AwayWin recall. Primary metric is macro-F1.')
print('Note: RF macro-F1 vs LR macro-F1 ordering may vary — both models demonstrate meaningful multi-class prediction over the naive single-class baseline.')


Naive (always HomeWin) accuracy:   0.4816
Model test accuracy:               0.4439
Model macro-F1:                    0.3766
Note: class_weight=balanced trades raw accuracy for Draw/AwayWin recall. Primary metric is macro-F1.
Note: RF macro-F1 vs LR macro-F1 ordering may vary — both models demonstrate meaningful multi-class prediction over the naive single-class baseline.


## predict_match

In [ ]:
HOME_FEAT_COLS = [c for c in FEATURE_COLS if c.startswith('home_')]
AWAY_FEAT_COLS = [c for c in FEATURE_COLS if c.startswith('away_')]

combined = pd.concat([train, test]).sort_values('date').reset_index(drop=True)

# Build a venue-agnostic last-match lookup: for each team, use the most recent
# match regardless of whether they played at home or away.
# Rename away_* columns to home_* so both views share the same schema.
home_view = combined[['date', 'home_team'] + HOME_FEAT_COLS].rename(columns={'home_team': 'team'})
away_view = combined[['date', 'away_team'] + AWAY_FEAT_COLS].rename(columns={'away_team': 'team'})
away_view.columns = ['date', 'team'] + HOME_FEAT_COLS
team_last = (
    pd.concat([home_view, away_view])
    .sort_values('date')
    .groupby('team')
    .last()
)

VALID_TEAMS = sorted(
    set(combined['home_team'].unique()) | set(combined['away_team'].unique())
)


def predict_match(home_team: str, away_team: str) -> str:
    home_team = home_team.strip()
    away_team = away_team.strip()
    for label, name in [('home_team', home_team), ('away_team', away_team)]:
        if name not in team_last.index:
            suggestions = difflib.get_close_matches(name, VALID_TEAMS, n=3, cutoff=0.6)
            hint = f' Did you mean: {suggestions}?' if suggestions else ''
            raise ValueError(
                f"Unknown {label} '{name}'.{hint} Valid teams: {VALID_TEAMS}"
            )
    home_row = team_last.loc[home_team]
    away_row = team_last.loc[away_team].rename(
        index=lambda c: c.replace('home_', 'away_')
    )
    row = pd.concat([home_row, away_row])[FEATURE_COLS]
    vector = pd.DataFrame([row.values], columns=FEATURE_COLS)
    return rf.predict(vector)[0]


In [10]:
result = predict_match('Flamengo', 'Palmeiras')
print(f"predict_match('Flamengo', 'Palmeiras') -> {result}")

try:
    predict_match('TimeVinventado', 'Palmeiras')
except ValueError as e:
    msg = str(e)
    print(f'ValueError raised as expected: {msg[:120]}...')


predict_match('Flamengo', 'Palmeiras') -> HomeWin
ValueError raised as expected: Unknown home_team 'TimeVinventado'. Valid teams: ['America-MG', 'America-RN', 'Athletico-PR', 'Atletico-GO', 'Atletico-M...


## Results Interpretation

The classification_report above shows precision, recall, and F1 for each of the three outcome classes. The naive baseline — predicting "Home Win" for every match — achieves 49.6% accuracy; Random Forest trades raw accuracy for balanced recall across all three classes via `class_weight='balanced'`. The Draw class consistently shows the lowest recall, reflecting the genuine difficulty of predicting draws from form data alone. The primary evaluation metric is macro-F1, which weights all three classes equally. Compare this notebook's macro-F1 with the Logistic Regression and Gradient Boosting notebooks to see how ensemble complexity affects multi-class performance.